In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [26]:
df = pd.read_csv("passwords_dataset.csv")

In [27]:
df.head()

,Password,Has Lowercase,Has Uppercase,Has Special Character,Length,Strength
0,<%r?.,True,False,True,5,Weak
1,l(d_l,True,False,True,5,Weak
2,"|+Z)kDTRYo:q{""(",True,True,True,15,Strong
3,gwcNB[oS5!n%OPJ,True,True,True,15,Strong
4,^vXjCCP6,True,True,True,8,Strong


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Password               10000 non-null  object
 1   Has Lowercase          10000 non-null  bool  
 2   Has Uppercase          10000 non-null  bool  
 3   Has Special Character  10000 non-null  bool  
 4   Length                 10000 non-null  int64 
 5   Strength               10000 non-null  object
dtypes: bool(3), int64(1), object(2)
memory usage: 263.8+ KB


In [29]:
df.duplicated().sum()

0

In [30]:
def extract_features(password):
    length = len(password)
    
    upper = sum(1 for c in password if c.isupper())
    lower = sum(1 for c in password if c.islower())
    digits = sum(1 for c in password if c.isdigit())
    special = len(re.findall(r'[^a-zA-Z0-9]', password))
    
    prob = [float(password.count(c)) / length for c in set(password)]
    entropy = -sum([p * np.log2(p) for p in prob])
    
    return pd.Series([length, upper, lower, digits, special, entropy])

In [31]:
df[['length','upper_count','lower_count','digit_count','special_count','entropy']] = df['Password'].apply(extract_features)

In [32]:
# Drop original columns that are now redundant.
# 'Length' was in the CSV, but extract_features() already computes 'length'.
# Keeping both would give the scaler 7 features while prediction supplies only 6.
df = df.drop(columns=[
    'Password',
    'Has Lowercase',
    'Has Uppercase',
    'Has Special Character',
    'Length'          # KEY FIX: must drop the CSV 'Length' column
])

In [36]:
X = df.drop(columns=['Strength'])
y = df['Strength']

In [37]:
le = LabelEncoder()
y = le.fit_transform(y)

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [39]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [40]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

In [41]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [42]:
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/15


200/200 [==============================] - 1s 2ms/step - loss: 0.4567 - accuracy: 0.8358 - val_loss: 0.2690 - val_accuracy: 0.9137
Epoch 2/15
200/200 [==============================] - 0s 1ms/step - loss: 0.2261 - accuracy: 0.9241 - val_loss: 0.1837 - val_accuracy: 0.9450
Epoch 3/15
200/200 [==============================] - 0s 1ms/step - loss: 0.1639 - accuracy: 0.9534 - val_loss: 0.1327 - val_accuracy: 0.9619
Epoch 4/15
200/200 [==============================] - 0s 1ms/step - loss: 0.1202 - accuracy: 0.9655 - val_loss: 0.0969 - val_accuracy: 0.9744
Epoch 5/15
200/200 [==============================] - 0s 1ms/step - loss: 0.0911 - accuracy: 0.9772 - val_loss: 0.0696 - val_accuracy: 0.9837
Epoch 6/15
200/200 [==============================] - 0s 1ms/step - loss: 0.0664 - accuracy: 0.9878 - val_loss: 0.0511 - val_accuracy: 0.9887
Epoch 7/15
200/200 [==============================] - 0s 1ms/step - loss: 0.0508 - accuracy: 0.9909 - val_loss: 0.0376 - val_accuracy: 0.9956
Epoc

In [43]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

63/63 [==============================] - 0s 846us/step - loss: 0.0102 - accuracy: 0.9995
Test Accuracy: 0.9994999766349792


In [44]:
y_pred = model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

63/63 [==============================] - 0s 1ms/step
[[ 370    0    0]
 [   0 1201    0]
 [   0    1  428]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       370
           1       1.00      1.00      1.00      1201
           2       1.00      1.00      1.00       429

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [45]:
def predict_password_strength(password):
    features = extract_features(password)
    # Convert to numpy array & reshape -> avoids sklearn feature-name warning
    features = np.array(features).reshape(1, -1)
    features = scaler.transform(features)
    
    pred = model.predict(features, verbose=0)
    label = np.argmax(pred)
    
    return le.inverse_transform([label])[0]

In [ ]:
print(predict_password_strength("P@ss123"))
print(predict_password_strength("weakpass"))
print(predict_password_strength("Xy@9$Lm#2024"))